# Exercise 1 - operational vs analytical


In [1]:
from faker.providers import BaseProvider
from faker import Faker
from dataclasses import dataclass
from datetime import datetime

fake = Faker()
fake.seed_instance(0)
name_pool = [fake.name() for _ in range(5)]


@dataclass
class Transaction:
    id: int
    name: str
    datetime: datetime
    amount: float


class TransactionProvider(BaseProvider):
    __provider__ = "transaction"

    def transaction(self) -> Transaction:
        return Transaction(
            fake.unique.uuid4(),
            fake.random_element(elements=name_pool),
            fake.date_time_this_century(),
            fake.pyfloat(min_value=-120, max_value=100),
        )


fake.add_provider(TransactionProvider)
N = int(500)
rowbased = [fake.transaction() for _ in range(N)]

In [2]:
rowbased

[Transaction(id='88561712-e8e5-416a-bcbd-04c340212ef7', name='Peter Montgomery', datetime=datetime.datetime(2023, 7, 16, 7, 49, 53, 482969), amount=-94.9566),
 Transaction(id='af19922a-d9b8-4714-a61a-441c12e0c8b2', name='Elizabeth Woods', datetime=datetime.datetime(2012, 4, 26, 8, 31, 22, 662032), amount=-29.55),
 Transaction(id='e9bb17bc-a3f2-49bf-9c63-16b950f24455', name='Jorge Sullivan', datetime=datetime.datetime(2025, 3, 21, 10, 6, 12, 391683), amount=-6.699749),
 Transaction(id='eb2083e6-ce16-4dba-8ff1-8e0242af9fc3', name='Peter Montgomery', datetime=datetime.datetime(2023, 11, 25, 2, 34, 36, 960632), amount=65.51),
 Transaction(id='ab0c1681-c8f8-43d0-9329-0a4cb5d32b16', name='Norma Fisher', datetime=datetime.datetime(2015, 12, 19, 3, 58, 13, 733886), amount=-57.765752),
 Transaction(id='101fbccc-ded7-43e8-b421-eaeb534097ca', name='Jorge Sullivan', datetime=datetime.datetime(2023, 12, 4, 20, 53, 21, 882895), amount=-58.2334),
 Transaction(id='1759edc3-72ae-4244-8b01-63c1cd9d2b7d'

In [3]:
columnar = {
    "id": [t.id for t in rowbased],
    "name": [t.name for t in rowbased],
    "datetime": [t.datetime for t in rowbased],
    "amount": [t.amount for t in rowbased],
}

Calculate the maximal transaction amount in the row-based approach.


In [4]:
%%timeit

max(rowbased, key=lambda t: t.amount)

22.4 μs ± 1.3 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [5]:
%%timeit

max([row.amount for row in rowbased])

8.78 μs ± 390 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [6]:
%%timeit

max_amount = float("-inf")
for row in rowbased:
    if row.amount > max_amount:
        max_amount = row.amount
max_amount

6.63 μs ± 299 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


Calculate the sum of all transactions by Jorge Sullivan in the row-based approach.


In [20]:
%%timeit

sum([row.amount for row in rowbased if row.name == "Jorge Sullivan"])

6.93 μs ± 181 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [13]:
%%timeit

sum_value = 0
for t in rowbased:
    sum_value += t.amount
sum_value

11.4 μs ± 451 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


Calculate the maximum transaction amount in the columnar approach


In [14]:
%%timeit

max(columnar["amount"])

2.68 μs ± 91.8 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


Calculate the sum of all transactions by Jorge Sullivan in the columnar approach.


In [ ]:
%%timeit

# This is slower than row-based in pure python because :
# - zip adds overhead
# - still iterating over rows, negating the benefit of columnar storage
# The advantages of columnar storage for analytical queries become more apparent when using vectorized operations (numpy, pandas, polars, etc.)
sum(amount for amount, name in zip(columnar["amount"], columnar["name"]) if name == "Jorge Sullivan")

12.2 μs ± 201 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


Retrieve all information for the transaction with ID `989bb030-86c4-4b06-86a8-e22fd57caf0d`, both in the rowbased and in the columnar approach.


In [25]:
%%timeit

next(filter(lambda t: t.id == "989bb030-86c4-4b06-86a8-e22fd57caf0d", rowbased))

5.76 μs ± 142 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [27]:
%%timeit

[t for t in rowbased if t.id == "989bb030-86c4-4b06-86a8-e22fd57caf0d"][0]

6.92 μs ± 112 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [ ]:
%%timeit

# Also this is a contraditcion in pure python against actual database engines.
# Transactional databases are optimized with indexes to retreive rows by those indexed column values.
# Here in pure python we do not have that, but we can use the list.index() mehtod which seems to be actually faster
# than pure python loops in the rowbased approach.

idx = [columnar["id"].index("989bb030-86c4-4b06-86a8-e22fd57caf0d")][0]

Transaction(
    id=columnar["id"][idx],
    name=columnar["name"][idx],
    datetime=columnar["datetime"][idx],
    amount=columnar["amount"][idx],
)

2.18 μs ± 79.1 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
